# Medical Cost Personal Datasets

**Q: Perform Regression using NN and Decision Tree**

In [83]:
import pandas as pd
import os
from constants import *

df = pd.read_csv(os.path.join(DATASETS_DIR, "insurance_ml.csv"))
print(df.shape)
print(df.dtypes)
df.head()

(1338, 7)
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [84]:
from sklearn.model_selection import train_test_split

X = df.drop('charges', axis=1)
y = df['charges']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.10 / 0.80), random_state=RANDOM_SEED
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (936, 6)
X_val shape: (134, 6)
X_test shape: (268, 6)


In [85]:
from sklearn.preprocessing import LabelEncoder

le_sex = LabelEncoder()
X_train['sex'] = le_sex.fit_transform(X_train['sex'])
X_val['sex'] = le_sex.transform(X_val['sex'])
X_test['sex'] = le_sex.transform(X_test['sex'])

le_smoker = LabelEncoder()
X_train['smoker'] = le_smoker.fit_transform(X_train['smoker'])
X_val['smoker'] = le_smoker.transform(X_val['smoker'])
X_test['smoker'] = le_smoker.transform(X_test['smoker'])

X_train.head()

,age,sex,bmi,children,smoker,region
335,64,1,34.500,0,0,southwest
1169,37,0,34.105,1,0,northwest
419,63,0,26.980,0,1,northwest
516,20,1,35.310,1,0,southeast
420,64,1,33.880,0,1,southeast


In [86]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
onehot_encoder.fit(X_train[['region']])

for split_df in [X_train, X_val, X_test]:
    region_encoded = onehot_encoder.transform(split_df[['region']])
    region_encoded_df = pd.DataFrame(
        region_encoded,
        columns=onehot_encoder.get_feature_names_out(['region']),
        index=split_df.index
    )
    split_df.drop('region', axis=1, inplace=True)
    split_df[region_encoded_df.columns] = region_encoded_df

X_train.head()

,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
335,64,1,34.500,0,0,0.0,0.0,0.0,1.0
1169,37,0,34.105,1,0,0.0,1.0,0.0,0.0
419,63,0,26.980,0,1,0.0,1.0,0.0,0.0
516,20,1,35.310,1,0,0.0,0.0,1.0,0.0
420,64,1,33.880,0,1,0.0,0.0,1.0,0.0


In [87]:
from sklearn.preprocessing import StandardScaler
import joblib

numeric_cols = ['age', 'bmi', 'children']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

joblib.dump(scaler, os.path.join(MODELS_DIR, "insurance_scaler.pkl"))
print("Scaler saved")

X_train.head()

Scaler saved


,age,sex,bmi,children,smoker,region_northeast,region_northwest,region_southeast,region_southwest
335,1.736318,1,0.664310,-0.908294,0,0.0,0.0,0.0,1.0
1169,-0.170192,0,0.599178,-0.093181,0,0.0,1.0,0.0,0.0
419,1.665706,0,-0.575679,-0.908294,1,0.0,1.0,0.0,0.0
516,-1.370586,1,0.797873,-0.093181,0,0.0,0.0,1.0,0.0
420,1.736318,1,0.562077,-0.908294,1,0.0,0.0,1.0,0.0


In [88]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

xgb_model_path = os.path.join(MODELS_DIR, "insurance_xgb.pkl")

if os.path.exists(xgb_model_path):
    xgb_model = joblib.load(xgb_model_path)
    print("Loaded existing XGBoost model.")
else:
    xgb_model = XGBRegressor(random_state=RANDOM_SEED, max_depth=5, n_estimators=100)
    xgb_model.fit(X_train, y_train)
    joblib.dump(xgb_model, xgb_model_path)
    print("XGBoost model saved to", xgb_model_path)

xgb_val_preds = xgb_model.predict(X_val)
xgb_val_mse = mean_squared_error(y_val, xgb_val_preds)
xgb_val_r2 = r2_score(y_val, xgb_val_preds)
print(f"XGBoost (Validation) → MSE: {xgb_val_mse:.2f}, R²: {xgb_val_r2:.4f}")

Loaded existing XGBoost model.
XGBoost (Validation) → MSE: 33868192.17, R²: 0.8141


In [89]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau
import joblib
import os

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled = y_scaler.transform(y_val.values.reshape(-1, 1)).flatten()

joblib.dump(y_scaler, os.path.join(MODELS_DIR, "insurance_y_scaler.pkl"))

tf_model_path = os.path.join(MODELS_DIR, "insurance_linear_regression_tf.keras")

if os.path.exists(tf_model_path):
    lr_model_tf = tf.keras.models.load_model(tf_model_path)
    print("Loaded existing TensorFlow model.")
else:
    tf.random.set_seed(RANDOM_SEED)

    lr_model_tf = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(1)
    ])

    lr_model_tf.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss="mse"
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=20,
        restore_best_weights=True
    )

    lr_damping = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-6
    )

    history = lr_model_tf.fit(
        X_train.values, y_train_scaled,
        validation_data=(X_val.values, y_val_scaled),
        epochs=1000,
        batch_size=len(X_train),
        callbacks=[early_stop, lr_damping],
        verbose=0
    )

    print(f"Stopped at epoch {len(history.history['loss'])} (out of 1000 max)")
    for i in range(0, len(history.history['loss']), 100):
        print(f"Epoch {i + 1} | Train Loss: {history.history['loss'][i]:.4f} "
              f"| Val Loss: {history.history['val_loss'][i]:.4f}")

    lr_model_tf.save(tf_model_path)
    print("Model saved to", tf_model_path)

Stopped at epoch 838 (out of 1000 max)
Epoch 1 | Train Loss: 2.4573 | Val Loss: 2.7580
Epoch 101 | Train Loss: 0.7369 | Val Loss: 0.9260
Epoch 201 | Train Loss: 0.4258 | Val Loss: 0.5397
Epoch 301 | Train Loss: 0.3088 | Val Loss: 0.3795
Epoch 401 | Train Loss: 0.2747 | Val Loss: 0.3208
Epoch 501 | Train Loss: 0.2672 | Val Loss: 0.3011
Epoch 601 | Train Loss: 0.2660 | Val Loss: 0.2947
Epoch 701 | Train Loss: 0.2658 | Val Loss: 0.2926
Epoch 801 | Train Loss: 0.2658 | Val Loss: 0.2924
Model saved to D:/CitrusBits/pythonic-rebirth\models\insurance_linear_regression_tf.keras


In [90]:
val_preds_scaled = lr_model_tf.predict(X_val.values).flatten()
val_preds_tf = y_scaler.inverse_transform(val_preds_scaled.reshape(-1, 1)).flatten()

lr_val_mse_tf = mean_squared_error(y_val, val_preds_tf)
lr_val_r2_tf = r2_score(y_val, val_preds_tf)

print(f"TensorFlow Linear Regression (Validation) → MSE: {lr_val_mse_tf:.2f}, R²: {lr_val_r2_tf:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
TensorFlow Linear Regression (Validation) → MSE: 40575156.10, R²: 0.7773


In [91]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

dt_model_path = os.path.join(MODELS_DIR, "insurance_decision_tree.pkl")

if os.path.exists(dt_model_path):
    dt_model = joblib.load(dt_model_path)
    print("Loaded existing Decision Tree model.")
else:
    dt_model = DecisionTreeRegressor(random_state=RANDOM_SEED, max_depth=5)
    dt_model.fit(X_train, y_train)
    joblib.dump(dt_model, dt_model_path)
    print("Decision Tree model saved to", dt_model_path)

dt_val_preds = dt_model.predict(X_val)
dt_val_mse = mean_squared_error(y_val, dt_val_preds)
dt_val_r2 = r2_score(y_val, dt_val_preds)
print(f"Decision Tree (Validation) → MSE: {dt_val_mse:.2f}, R²: {dt_val_r2:.4f}")

Loaded existing Decision Tree model.
Decision Tree (Validation) → MSE: 26989461.41, R²: 0.8519


In [92]:
print(f"{'Model':<25}{'MSE':<15}{'R²':<10}")
print(f"{'TF Linear Regression':<25}{lr_val_mse_tf:<15.2f}{lr_val_r2_tf:<10.4f}")
print(f"{'Decision Tree':<25}{dt_val_mse:<15.2f}{dt_val_r2:<10.4f}")

Model                    MSE            R²        
TF Linear Regression     40575156.10    0.7773    
Decision Tree            26989461.41    0.8519    


In [93]:
import numpy as np

print(f"TF Linear Regression RMSE: ${np.sqrt(lr_val_mse_tf):.2f}")
print(f"Decision Tree RMSE: ${np.sqrt(dt_val_mse):.2f}")

TF Linear Regression RMSE: $6369.86
Decision Tree RMSE: $5195.14


In [94]:
val_preds_scaled_test = lr_model_tf.predict(X_test.values).flatten()
val_preds_tf_test = y_scaler.inverse_transform(val_preds_scaled_test.reshape(-1, 1)).flatten()

lr_test_mse_tf = mean_squared_error(y_test, val_preds_tf_test)
lr_test_r2_tf = r2_score(y_test, val_preds_tf_test)
print(f"TensorFlow Linear Regression (Test) → MSE: {lr_test_mse_tf:.2f}, R²: {lr_test_r2_tf:.4f}")

dt_test_preds = dt_model.predict(X_test)
dt_test_mse = mean_squared_error(y_test, dt_test_preds)
dt_test_r2 = r2_score(y_test, dt_test_preds)
print(f"Decision Tree (Test) → MSE: {dt_test_mse:.2f}, R²: {dt_test_r2:.4f}")

print(f"\n{'Model':<25}{'Test MSE':<18}{'Test R²':<10}")
print(f"{'TF Linear Regression':<25}{lr_test_mse_tf:<18.2f}{lr_test_r2_tf:<10.4f}")
print(f"{'Decision Tree':<25}{dt_test_mse:<18.2f}{dt_test_r2:<10.4f}")

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
TensorFlow Linear Regression (Test) → MSE: 33865561.86, R²: 0.7819
Decision Tree (Test) → MSE: 25985425.65, R²: 0.8326

Model                    Test MSE          Test R²   
TF Linear Regression     33865561.86       0.7819    
Decision Tree            25985425.65       0.8326    


In [95]:
xgb_test_preds = xgb_model.predict(X_test)
xgb_test_mse = mean_squared_error(y_test, xgb_test_preds)
xgb_test_r2 = r2_score(y_test, xgb_test_preds)
print(f"XGBoost (Test) → MSE: {xgb_test_mse:.2f}, R²: {xgb_test_r2:.4f}")

print(f"\n{'Model':<25}{'Test MSE':<18}{'Test R²':<10}")
print(f"{'TF Linear Regression':<25}{lr_test_mse_tf:<18.2f}{lr_test_r2_tf:<10.4f}")
print(f"{'Decision Tree':<25}{dt_test_mse:<18.2f}{dt_test_r2:<10.4f}")
print(f"{'XGBoost':<25}{xgb_test_mse:<18.2f}{xgb_test_r2:<10.4f}")

XGBoost (Test) → MSE: 23438440.48, R²: 0.8490

Model                    Test MSE          Test R²   
TF Linear Regression     33865561.86       0.7819    
Decision Tree            25985425.65       0.8326    
XGBoost                  23438440.48       0.8490    
